
# 01. DBAASP MIC Dataset Construction

## Objective

Construct EC and SA peptide MIC regression datasets from DBAASP activity records.

## Workflow

1. Load and validate raw DBAASP data.
2. Prepare peptide sequences and MIC records.
3. Parse MIC values and convert units to µM.
4. Exclude censored MIC values (`>`, `<`, `≥`, `≤`).
5. Estimate range-based MIC values using the geometric mean.
6. Aggregate MIC records using the median.
7. Construct EC and SA regression datasets.
8. Generate reproducible Train/Test splits.
9. Save intermediate and final datasets.

## Principles

- Preserve original data.
- Implement data processing in reusable Python modules.
- Execute and inspect each stage in this notebook.
- Use random state 42 for reproducible data splitting.
- Save all intermediate and final datasets.


In [2]:

import pandas as pd

from configs.common import DATA_DIR

# Raw data paths
RAW_DIR = DATA_DIR / "raw"

peptides_path = RAW_DIR / "DBAASP_peptides_raw.csv"
activities_path = RAW_DIR / "DBAASP_activities_raw.csv"

# Load raw datasets
df_peptides = pd.read_csv(peptides_path)
df_activities = pd.read_csv(activities_path)

# Check dataset structure
print(f"Peptides: {df_peptides.shape}")
print(f"Activities: {df_activities.shape}")

print("\nActivity columns:")
print(df_activities.columns.tolist())

# Preview
display(df_activities.head())


Peptides: (25542, 14)
Activities: (197338, 31)

Activity columns:
['peptide_id', 'dbaasp_id', 'sequence', 'sequence_length', 'activity', 'activityMeasureValue', 'cfu', 'cfuGroup', 'concentration', 'id', 'ionicStrength', 'note', 'peptideId', 'ph', 'reference', 'saltType', 'complexity.description', 'complexity.name', 'activityMeasureGroup.name', 'medium.description', 'medium.name', 'targetSpecies.name', 'unit.description', 'unit.name', 'cfuGroup.name', 'targetSpecies.description', 'activityMeasureGroup', 'unit', 'medium', 'activityMeasureGroup.description', 'targetSpecies']


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_24292\2711028975.py:13: DtypeWarning: Columns (6,13,29) have mixed types. Specify dtype option on import or set low_memory=False.
  df_activities = pd.read_csv(activities_path)


,peptide_id,dbaasp_id,sequence,sequence_length,activity,activityMeasureValue,cfu,cfuGroup,concentration,id,...,targetSpecies.name,unit.description,unit.name,cfuGroup.name,targetSpecies.description,activityMeasureGroup,unit,medium,activityMeasureGroup.description,targetSpecies
0,1,DBAASPR_1,NaN,0.0,0.261290,MIC,NaN,NaN,14.5,20652,...,Escherichia coli ATCC 25922,NaN,µM,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,DBAASPR_1,NaN,0.0,0.504560,MIC,NaN,NaN,28.0,20653,...,Staphylococcus aureus ATCC 25923,NaN,µM,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,DBAASPR_1,NaN,0.0,0.504560,MIC,NaN,NaN,28.0,20654,...,Pseudomonas aeruginosa ATCC 27853,NaN,µM,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,DBAASPR_1,NaN,0.0,0.261290,MIC,NaN,NaN,14.5,20655,...,Enterococcus faecalis ATCC 29212,NaN,µM,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1,DBAASPR_1,NaN,0.0,0.210474,MIC,5E5,NaN,0.73-11.68,66939,...,Escherichia coli,NaN,µM,1E5 - 1E6,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:

from src.dataset.preparation import inspect_raw_data

# Inspect raw activities
summary = inspect_raw_data(df_activities)

# Display results
for key, value in summary.items():
    print(f"{key}: {value:,}")


Samples: 197,338
Columns: 31
Missing_Sequence: 4,572
Missing_Activity: 0
Duplicate_Rows: 0


In [7]:

import importlib
import src.dataset.preparation as preparation

# Reload module
importlib.reload(preparation)

# Select MIC records
df_mic = preparation.select_mic_records(df_activities)

# Save intermediate data
output_dir = DATA_DIR / "intermediate"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "DBAASP_MIC_selected.csv"

df_mic.to_csv(output_path, index=False)

# Verify results
print(f"MIC records: {len(df_mic):,}")
print(f"Missing sequences: {df_mic['sequence'].isna().sum():,}")
print(f"Saved: {output_path.relative_to(DATA_DIR)}")

display(
    df_mic[
        ["sequence", "activity", "unit.name", "targetSpecies.name"]
    ].head(10)
)


MIC records: 142,169
Missing sequences: 3,124
Saved: intermediate\DBAASP_MIC_selected.csv


,sequence,activity,unit.name,targetSpecies.name
0,NaN,0.261290,µM,Escherichia coli ATCC 25922
1,NaN,0.504560,µM,Staphylococcus aureus ATCC 25923
2,NaN,0.504560,µM,Pseudomonas aeruginosa ATCC 27853
3,NaN,0.261290,µM,Enterococcus faecalis ATCC 29212
4,NaN,0.210474,µM,Escherichia coli
6,NaN,0.421127,µM,Klebsiella pneumoniae
8,NaN,0.842075,µM,Serratia marcescens
10,NaN,0.842976,µM,Stenotrophomonas maltophilia
12,NaN,0.842075,µM,Acinetobacter baumannii
14,NaN,0.842075,µM,Pseudomonas aeruginosa


In [11]:

from src.dataset.mic_parser import parse_mic_dataframe

# Parse MIC values
df_parsed = parse_mic_dataframe(df_mic)

# Summary
print("MIC Parsing Results")
display(
    df_parsed["mic_type"]
    .value_counts(dropna=False)
    .reset_index(name="Count")
)

# Preview
display(
    df_parsed[
        [
            "concentration",
            "mic_parsed",
            "mic_type",
            "unit.name"
        ]
    ].head(20)
)

# Save intermediate data
output_path = (
    DATA_DIR / "intermediate" /
    "DBAASP_MIC_parsed.csv"
)

df_parsed.to_csv(output_path, index=False)

print(f"\n[OK] Saved: {output_path.relative_to(DATA_DIR)}")


MIC Parsing Results


,mic_type,Count
0,exact,100246
1,censored,30756
2,range_imputed,7531
3,mean,3563
4,invalid,52
5,missing,19
6,unknown,2


,concentration,mic_parsed,mic_type,unit.name
0,14.5,14.500000,exact,µM
1,28.0,28.000000,exact,µM
2,28.0,28.000000,exact,µM
3,14.5,14.500000,exact,µM
4,0.73-11.68,2.920000,range_imputed,µM
6,0.73-23.37,4.130387,range_imputed,µM
8,1.46-46.73,8.259891,range_imputed,µM
10,2.92-46.78,11.687498,range_imputed,µM
12,2.92-46.73,11.681250,range_imputed,µM
14,2.92-46.73,11.681250,range_imputed,µM



[OK] Saved: intermediate\DBAASP_MIC_parsed.csv



## MIC Unit Conversion

### Objective
Standardize peptide MIC values to µM.

### Procedure
1. Retain sequences containing 20 standard amino acids.
2. Calculate molecular weights using Biopython.
3. Standardize MIC unit names.
4. Convert MIC values to µM.
5. Save the converted dataset.

### Conversion
- µM: Retain original values.
- µg/mL: MIC (µM) = MIC (µg/mL) × 1000 / MW (Da).
- Unsupported units: Exclude from conversion.
- Censored MIC values: Keep as missing and exclude during filtering.

Molecular weights are calculated from unmodified peptide sequences.


In [14]:
from src.dataset.mic_conversion import convert_mic_units

# Convert MIC units
df_converted = convert_mic_units(df_parsed)

# Check results
print(f"Parsed MIC records: {len(df_parsed):,}")
print(f"Standard AA records: {len(df_converted):,}")
print(f"Converted MIC records: {df_converted['mic_uM'].notna().sum():,}")
print(f"Unconverted records: {df_converted['mic_uM'].isna().sum():,}")

# Unit distribution
display(
    df_converted["unit_standardized"]
    .value_counts(dropna=False)
    .to_frame("Count")
)

# Conversion preview
display(
    df_converted[
        [
            "sequence",
            "concentration",
            "mic_type",
            "unit.name",
            "molecular_weight",
            "mic_uM"
        ]
    ].head(20)
)

# Save intermediate dataset
output_path = (
    DATA_DIR / "intermediate" /
    "DBAASP_MIC_converted.csv"
)

df_converted.to_csv(output_path, index=False)

print(f"\\n[OK] Saved: {output_path.relative_to(DATA_DIR)}")


c:\Users\jinwo\anaconda3\envs\bert-ampep60\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Calculating molecular weights: 100%|██████████| 13690/13690 [00:00<00:00, 327483.77it/s]

Parsed MIC records: 142,169
Standard AA records: 111,256
Converted MIC records: 86,259
Unconverted records: 24,997


,Count
unit_standardized,
um,63158
ug/ml,48072
<NA>,26


,sequence,concentration,mic_type,unit.name,molecular_weight,mic_uM
66,KVVVKWVVKVVK,>100,censored,µM,1410.8314,NaN
67,KVVVKWVVKVVK,>100,censored,µM,1410.8314,NaN
68,KVVVKWVVKVVK,>100,censored,µM,1410.8314,NaN
69,KVVVKWVVKVVK,>100,censored,µM,1410.8314,NaN
70,KVVVKWVVKVVK,>100,censored,µM,1410.8314,NaN
71,KVVVKWVVKVVK,3.125,exact,µM,1410.8314,3.125000
72,KVVVKWVVKVVK,3.125,exact,µM,1410.8314,3.125000
73,KVVVKWVVKVVK,>100,censored,µM,1410.8314,NaN
74,KVVVKWVVKVVK,>100,censored,µM,1410.8314,NaN
75,KVVVKWVVKVVK,>100,censored,µM,1410.8314,NaN


\n[OK] Saved: intermediate\DBAASP_MIC_converted.csv


In [16]:
from src.dataset.mic_filter import filter_mic_records

# Filter valid MIC records
df_filtered = filter_mic_records(df_converted)

# Check filtering
print(f"Before filtering: {len(df_converted):,}")
print(f"After filtering: {len(df_filtered):,}")
print(f"Excluded: {len(df_converted) - len(df_filtered):,}")

# MIC type distribution
display(
    df_filtered["mic_type"]
    .value_counts()
    .to_frame("Count")
)

# Validate results
assert df_filtered["mic_type"].isin(
    ["exact", "mean", "range_imputed"]
).all()

assert df_filtered["mic_uM"].notna().all()
assert (df_filtered["mic_uM"] > 0).all()

# Save intermediate data
output_path = (
    DATA_DIR / "intermediate" /
    "DBAASP_MIC_filtered.csv"
)

df_filtered.to_csv(output_path, index=False)

print(f"\n[OK] Saved: {output_path.relative_to(DATA_DIR)}")


Before filtering: 111,256
After filtering: 86,259
Excluded: 24,997


,Count
mic_type,
exact,78264
range_imputed,5072
mean,2923



[OK] Saved: intermediate\DBAASP_MIC_filtered.csv


In [18]:

from src.dataset.species import standardize_species

# Standardize species
df_species = standardize_species(df_filtered)

# Check results
print(f"MIC records: {len(df_species):,}")
print(
    "Original species names:",
    df_species["targetSpecies.name"].nunique()
)
print(
    "Standardized species:",
    df_species["species"].nunique()
)

# Preview
display(
    df_species[
        ["sequence", "targetSpecies.name", "species", "mic_uM"]
    ].head(20)
)

# Save intermediate dataset
output_path = (
    DATA_DIR / "intermediate" /
    "DBAASP_MIC_species.csv"
)

df_species.to_csv(output_path, index=False)

print(f"\n[OK] Saved: {output_path.relative_to(DATA_DIR)}")


MIC records: 86,259
Original species names: 4837
Standardized species: 678


,sequence,targetSpecies.name,species,mic_uM
71,KVVVKWVVKVVK,Candida albicans ATCC 10231,Candida albicans,3.125000
72,KVVVKWVVKVVK,Cryptococcus neoformans ATCC MYA-422,Cryptococcus neoformans,3.125000
78,LFIFFF,Staphylococcus aureus,Staphylococcus aureus,15.419972
80,LFIFFF,Streptococcus sanguinis,Streptococcus sanguinis,4.649296
81,RVKRVWPLVIRTVIAGYNLYRAIKKK,Salmonella enterica subsp. enterica serovar Ty...,Salmonella enterica,2.000000
82,RVKRVWPLVIRTVIAGYNLYRAIKKK,Listeria monocytogenes ATCC 19115,Listeria monocytogenes,2.000000
83,RVKRVWPLVIRTVIAGYNLYRAIKKK,Escherichia coli ATCC 25922,Escherichia coli,4.000000
84,RVKRVWPLVIRTVIAGYNLYRAIKKK,Escherichia coli ATCC 25922,Escherichia coli,1.590000
85,RVKRVWPLVIRTVIAGYNLYRAIKKK,Escherichia coli O157:H7 ATCC 700728,Escherichia coli,0.800000
86,RVKRVWPLVIRTVIAGYNLYRAIKKK,Salmonella enterica subsp. enterica serovar Ty...,Salmonella enterica,0.800000



[OK] Saved: intermediate\DBAASP_MIC_species.csv


In [20]:
from src.dataset.mic_aggregation import aggregate_mic

# Aggregate MIC records
df_aggregated = aggregate_mic(df_species)

# Check aggregation
print(f"Original MIC records: {len(df_species):,}")
print(f"Aggregated records: {len(df_aggregated):,}")
print(f"Unique peptides: {df_aggregated['sequence'].nunique():,}")
print(f"Unique species: {df_aggregated['species'].nunique():,}")

# Verify uniqueness
assert not df_aggregated.duplicated(
    subset=["sequence", "species"]
).any()

# Preview
display(df_aggregated.head(20))

# Save intermediate dataset
output_path = (
    DATA_DIR / "intermediate" /
    "DBAASP_MIC_aggregated.csv"
)

df_aggregated.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"\n[OK] Saved: {output_path.relative_to(DATA_DIR)}")


Original MIC records: 86,259
Aggregated records: 53,827
Unique peptides: 12,153
Unique species: 678


,sequence,species,original_strains,mic_uM,MIC_records,log10_MIC
0,A,Enterobacter sp,Enterobacter sp.,11.224201,1,1.050155
1,A,Enterococcus faecalis,Enterococcus faecalis,22.448402,1,1.351185
2,A,Escherichia coli,Escherichia coli NCIB 8879,11.224201,1,1.050155
3,A,Haemophilus influenzae,Haemophilus influenzae NCTC 4560,718.348875,1,2.856335
4,A,Klebsiella aerogenes,Klebsiella aerogenes,5.612101,1,0.749125
5,A,Salmonella typhimurium,Salmonella typhimurium,718.348875,1,2.856335
6,A,Serratia marcescens,Serratia marcescens ATCC 14756,89.793609,1,1.953245
7,A,Staphylococcus aureus,Staphylococcus aureus NCIB 8625,359.174438,1,2.555305
8,AA,Enterococcus faecalis,Enterococcus faecalis FS5,0.374599,1,-0.426433
9,AA,Escherichia coli,Escherichia coli NCIB 8879,3.121662,1,0.494386



[OK] Saved: intermediate\DBAASP_MIC_aggregated.csv


In [26]:
import numpy as np
from src.dataset.processing import process_dataset

# Process aggregated MIC dataset
df_processed = process_dataset(df_aggregated)

# Verify processing
print(f"Before processing: {len(df_aggregated):,}")
print(f"After processing : {len(df_processed):,}")
print(f"Excluded records : {len(df_aggregated) - len(df_processed):,}")

# Verify peptide length
assert df_processed["length"].between(5, 50).all()

# Verify standard pMIC
assert np.allclose(
    df_processed["pMIC"],
    6 - np.log10(df_processed["mic_uM"])
)

# Preview
display(df_processed.head(10))

# Save intermediate dataset
output_path = (
    DATA_DIR / "intermediate" /
    "DBAASP_MIC_processed.csv"
)

df_processed.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"\n[OK] Saved: {output_path.relative_to(DATA_DIR)}")


Before processing: 53,827
After processing : 51,502
Excluded records : 2,325


,sequence,species,original_strains,mic_uM,MIC_records,log10_MIC,length,pMIC
0,AAAAAAAIKMLMDLVNERIMALNKKAKK,Escherichia coli,Escherichia coli ATCC 25922,10.0,1,1.000000,28,5.000000
1,AAAAAAAIKMLMDLVNERIMALNKKAKK,Pseudomonas aeruginosa,Pseudomonas aeruginosa ATCC 27853,20.0,1,1.301030,28,4.698970
2,AAAAAAAIKMLMDLVNERIMALNKKAKK,Staphylococcus aureus,Staphylococcus aureus ATCC 29313,20.0,1,1.301030,28,4.698970
3,AAAAGSCVWGAVNYTSDCAAECKRRGYKGGHCGSFANVNCWCET,Fusarium culmorum,Fusarium culmorum IMI 180420,6.0,1,0.778151,44,5.221849
4,AAAAGSCVWGAVNYTSDCAAECKRRGYKGGHCGSFANVNCWCET,Fusarium haematococcum,Fusarium haematococcum 160.2.2,12.0,1,1.079181,44,4.920819
5,AAAAGSCVWGAVNYTSDCAAECKRRGYKGGHCGSFANVNCWCET,Micrococcus luteus,Micrococcus luteus A270,50.0,1,1.698970,44,4.301030
6,AAAAGSCVWGAVNYTSDCAAECKRRGYKGGHCGSFANVNCWCET,Neurospora crassa,Neurospora crassa CBS 327-54,3.0,1,0.477121,44,5.522879
7,AAAAGSCVWGAVNYTSDCAAECKRRGYKGGHCGSFANVNCWCRT,Bacillus megaterium,Bacillus megaterium,100.0,1,2.000000,44,4.000000
8,AAAAGSCVWGAVNYTSDCAAECKRRGYKGGHCGSFANVNCWCRT,Fusarium culmorum,Fusarium culmorum IMI 180420,6.0,1,0.778151,44,5.221849
9,AAAAGSCVWGAVNYTSDCAAECKRRGYKGGHCGSFANVNCWCRT,Fusarium haematococcum,Fusarium haematococcum 160.2.2,12.0,1,1.079181,44,4.920819



[OK] Saved: intermediate\DBAASP_MIC_processed.csv


In [28]:

from src.dataset.splitting import create_train_test

# Create EC and SA datasets
final_datasets = {}

for target in ["EC", "SA"]:
    train, test = create_train_test(
        df_processed,
        target=target
    )

    final_datasets[f"{target}_Train"] = train
    final_datasets[f"{target}_Test"] = test

    # Save final datasets
    train.to_csv(
        DATA_DIR / f"{target}_Train.csv",
        index=False,
        encoding="utf-8-sig"
    )

    test.to_csv(
        DATA_DIR / f"{target}_Test.csv",
        index=False,
        encoding="utf-8-sig"
    )

    # Verify results
    print(f"\\n[{target}]")
    print(f"Train: {len(train):,}")
    print(f"Test : {len(test):,}")
    print(f"Total: {len(train) + len(test):,}")

    print(
        "Sequence overlap:",
        len(
            set(train["sequence"])
            & set(test["sequence"])
        )
    )

print("\\n[OK] Final datasets saved.")


\n[EC]
Train: 7,583
Test : 843
Total: 8,426
Sequence overlap: 0
\n[SA]
Train: 6,592
Test : 733
Total: 7,325
Sequence overlap: 0
\n[OK] Final datasets saved.


In [29]:

import numpy as np
import pandas as pd

from configs.common import DATA_DIR

# Final dataset validation
summary = []

for target in ["EC", "SA"]:

    train = pd.read_csv(DATA_DIR / f"{target}_Train.csv")
    test = pd.read_csv(DATA_DIR / f"{target}_Test.csv")

    overlap = len(
        set(train["sequence"]) & set(test["sequence"])
    )

    for split, df in [("Train", train), ("Test", test)]:

        # Validate sequences
        valid_sequences = df["sequence"].str.fullmatch(
            r"[ACDEFGHIKLMNPQRSTVWY]+",
            na=False
        ).all()

        # Validate standard pMIC
        valid_pmic = np.allclose(
            df["pMIC"],
            6 - np.log10(df["mic_uM"])
        )

        # Validate MIC and length
        valid_length = df["length"].between(5, 50).all()
        valid_mic = (
            (df["mic_uM"] > 0)
            & (df["mic_uM"] <= 10000)
        ).all()

        summary.append({
            "Dataset": f"{target}_{split}",
            "Samples": len(df),
            "Missing": int(df.isna().sum().sum()),
            "Duplicate_Sequences": int(
                df["sequence"].duplicated().sum()
            ),
            "Valid_Sequences": bool(valid_sequences),
            "Valid_Length": bool(valid_length),
            "Valid_MIC": bool(valid_mic),
            "Valid_pMIC": bool(valid_pmic),
            "Train_Test_Overlap": overlap
        })

validation_df = pd.DataFrame(summary)

display(validation_df)

# Verify all conditions
assert validation_df["Valid_Sequences"].all()
assert validation_df["Valid_Length"].all()
assert validation_df["Valid_MIC"].all()
assert validation_df["Valid_pMIC"].all()
assert (validation_df["Train_Test_Overlap"] == 0).all()

print("\n[OK] Final dataset validation passed.")


,Dataset,Samples,Missing,Duplicate_Sequences,Valid_Sequences,Valid_Length,Valid_MIC,Valid_pMIC,Train_Test_Overlap
0,EC_Train,7583,0,0,True,True,True,True,0
1,EC_Test,843,0,0,True,True,True,True,0
2,SA_Train,6592,0,0,True,True,True,True,0
3,SA_Test,733,0,0,True,True,True,True,0



[OK] Final dataset validation passed.


In [32]:

import pandas as pd

from configs.common import DATA_DIR

# Define dataset processing stages
stages = [
    ("Initial DBAASP peptide entries", df_peptides),
    ("Antimicrobial activity records", df_activities),
    ("MIC records", df_mic),
    ("Standard amino acid filtering and unit conversion", df_converted),
    ("MIC filtering (censored and invalid values excluded)", df_filtered),
    ("MIC aggregation by peptide-species pair", df_aggregated),
    ("Peptide length and MIC range filtering", df_processed),
]

# Generate processing summary
summary = []

for stage, df in stages:
    summary.append({
        "Dataset processing stage": stage,
        "Records / pairs (n)": len(df),
        "Unique sequences (n)": df["sequence"].nunique()
    })

# Add final EC and SA datasets
for target in ["EC", "SA"]:

    train = final_datasets[f"{target}_Train"]
    test = final_datasets[f"{target}_Test"]

    combined = pd.concat(
        [train, test], ignore_index=True
    )

    for label, df in [
        (f"Final {target} dataset", combined),
        (f"{target} training set", train),
        (f"{target} test set", test)
    ]:
        summary.append({
            "Dataset processing stage": label,
            "Records / pairs (n)": len(df),
            "Unique sequences (n)": df["sequence"].nunique()
        })

# Create summary table
summary_df = pd.DataFrame(summary)

# Display summary
display(summary_df)

# Save summary
output_path = (
    DATA_DIR / "intermediate" /
    "DBAASP_Dataset_Processing_Summary.csv"
)

summary_df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"\n[OK] Saved: {output_path.relative_to(DATA_DIR)}")


,Dataset processing stage,Records / pairs (n),Unique sequences (n)
0,Initial DBAASP peptide entries,25542,21268
1,Antimicrobial activity records,197338,20869
2,MIC records,142169,16863
3,Standard amino acid filtering and unit conversion,111256,13690
4,MIC filtering (censored and invalid values exc...,86259,12153
5,MIC aggregation by peptide-species pair,53827,12153
6,Peptide length and MIC range filtering,51502,11770
7,Final EC dataset,8426,8426
8,EC training set,7583,7583
9,EC test set,843,843



[OK] Saved: intermediate\DBAASP_Dataset_Processing_Summary.csv


In [31]:

# Compare unique peptide sequences
peptide_sequences = set(
    df_peptides["sequence"].dropna().astype(str).str.strip()
)

activity_sequences = set(
    df_activities["sequence"].dropna().astype(str).str.strip()
)

# Compare sequence sets
only_peptides = peptide_sequences - activity_sequences
only_activities = activity_sequences - peptide_sequences
shared_sequences = peptide_sequences & activity_sequences

print(f"Peptide sequences  : {len(peptide_sequences):,}")
print(f"Activity sequences : {len(activity_sequences):,}")
print(f"Shared sequences   : {len(shared_sequences):,}")
print(f"Only in peptides   : {len(only_peptides):,}")
print(f"Only in activities : {len(only_activities):,}")


Peptide sequences  : 21,268
Activity sequences : 20,869
Shared sequences   : 20,869
Only in peptides   : 399
Only in activities : 0
